# Анализ временного ряда: Международные туристские прибытия в Испанию, 2010–2023

## 0. Импорт библиотек и настройка

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf as acf_func

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'legend.fontsize': 9.5,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.30,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

OUT_ASSETS = '../assets'
OUT_TABLES = '../tables'
OUT_GEN    = '../generated'
for d in [OUT_ASSETS, OUT_TABLES, OUT_GEN]:
    os.makedirs(d, exist_ok=True)

# ── Цветовая схема ──────────────────────────────────────────────────────────
C_GROWTH   = '#2C7BB6'
C_COVID    = '#D7191C'
C_RECOV    = '#1A9641'
C_MA3      = '#F4A460'
C_MA6      = '#8B4513'
C_MA12     = '#2C7BB6'
C_EXP_LOW  = '#9B59B6'
C_EXP_HIGH = '#E67E22'
C_LIN      = '#E74C3C'
C_QUAD     = '#2ECC71'
C_RAW      = '#2C3E50'
C_TREND    = '#2C7BB6'
C_SEAS     = '#E67E22'
C_RESID    = '#1A9641'

phase_color = {'growth': C_GROWTH, 'covid': C_COVID, 'recovery': C_RECOV}
phase_label = {
    'growth':   'Устойчивый рост, 2010–2019',
    'covid':    'COVID-19, 2020–2021',
    'recovery': 'Восстановление, 2022–2023',
}

months_ru      = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']
months_full_ru = ['Январь','Февраль','Март','Апрель','Май','Июнь',
                  'Июль','Август','Сентябрь','Октябрь','Ноябрь','Декабрь']

print("✓ Настройки загружены")


## 1. Данные

In [ ]:
raw_data = {
    2010: [2390, 2534, 3227, 3988, 3999, 5035, 7259, 7373, 5440, 4733, 2889, 3810],
    2011: [2563, 2763, 3470, 4239, 4531, 5453, 7445, 8057, 5869, 5051, 3108, 4145],
    2012: [2599, 2753, 3454, 4242, 4541, 5510, 7995, 7874, 6028, 5221, 3146, 4338],
    2013: [2722, 2896, 3649, 4414, 4605, 5780, 8232, 8494, 6471, 5562, 3332, 4504],
    2014: [2901, 3079, 3995, 4819, 5001, 6227, 8842, 9089, 6795, 5906, 3536, 4805],
    2015: [3093, 3378, 4109, 5223, 5442, 6373, 9121, 9461, 7112, 6113, 3816, 4974],
    2016: [3386, 3623, 4542, 5687, 6030, 7055,10223,10334, 7947, 6880, 4027, 5581],
    2017: [3655, 3951, 4964, 6240, 6396, 7774,11011,11401, 8699, 7329, 4430, 6017],
    2018: [3703, 4001, 5117, 6078, 6493, 7950,11130,11586, 8586, 7484, 4579, 6099],
    2019: [3735, 4047, 5108, 6371, 6461, 7993,11340,11709, 8713, 7602, 4464, 6131],
    2020: [2232, 2206,  263,   13,  131,  919, 2495, 3020, 2757, 2573, 1313, 1102],
    2021: [ 405,  437,  780, 1154, 1621, 3056, 5301, 6080, 4677, 3586, 1840, 2244],
    2022: [3213, 3413, 4275, 5515, 5593, 6861, 9683,10078, 7540, 6463, 3922, 5102],
    2023: [3866, 4061, 5024, 6399, 6621, 7938,11934,12159, 8940, 7293, 4600, 6261],
}

records = []
for y in sorted(raw_data):
    for m_idx, val in enumerate(raw_data[y]):
        records.append({'year': y, 'month': m_idx+1, 'arrivals': val})

df = pd.DataFrame(records)
df['date'] = pd.to_datetime({'year': df['year'], 'month': df['month'], 'day': 1})
df = df.sort_values('date').reset_index(drop=True)
df['t'] = np.arange(1, len(df)+1)
df['arrivals_m'] = df['arrivals'] / 1000.0

def phase(y):
    if y <= 2019: return 'growth'
    elif y <= 2021: return 'covid'
    return 'recovery'

df['phase'] = df['year'].apply(phase)

# ── Скользящие средние ─────────────────────────────────────────────────────
df['ma3'] = df['arrivals_m'].rolling(3).mean()
df['ma6'] = df['arrivals_m'].rolling(6).mean()

w12 = np.array([0.5]+[1]*11+[0.5]) / 12.0
arr = df['arrivals_m'].values
ma12 = np.full(len(arr), np.nan)
for i in range(6, len(arr)-6):
    ma12[i] = np.dot(w12, arr[i-6:i+7])
df['ma12'] = ma12

# ── Экспоненциальное сглаживание ───────────────────────────────────────────
def exp_smooth(series, alpha):
    s = np.full(len(series), np.nan)
    s[0] = series[0]
    for i in range(1, len(series)):
        s[i] = alpha * series[i] + (1-alpha)*s[i-1]
    return s

df['exp02'] = exp_smooth(df['arrivals_m'].values, 0.2)
df['exp06'] = exp_smooth(df['arrivals_m'].values, 0.6)
df['yoy']   = df.groupby('month')['arrivals_m'].pct_change() * 100

print(f"✓ Загружено наблюдений: {len(df)}")
df[['date','year','month','arrivals_m','phase']].head(12)


## 2. Аддитивная декомпозиция

In [ ]:
trend     = df['ma12'].values
detrended = df['arrivals_m'].values - trend

seas_coefs = {}
seas_ci    = {}   # half-width of 95% confidence interval
for m in range(1, 13):
    vals = detrended[df['month'] == m]
    vals = vals[~np.isnan(vals)]
    seas_coefs[m] = np.nanmean(vals)
    se = np.nanstd(vals, ddof=1) / np.sqrt(len(vals))
    seas_ci[m] = 1.96 * se

s_mean = np.mean(list(seas_coefs.values()))
for m in seas_coefs:
    seas_coefs[m] -= s_mean

df['seasonal'] = df['month'].map(seas_coefs)
df['residual'] = df['arrivals_m'] - df['ma12'] - df['seasonal']

print("Сезонные коэффициенты (тыс. чел.):")
for m in range(1, 13):
    print(f"  {months_full_ru[m-1]:12s}: {seas_coefs[m]*1000:+.0f}")


## 3. Тест на стационарность (ADF)

In [ ]:
series_raw = df['arrivals_m'].values
series_d1  = np.diff(series_raw)
series_sd  = series_raw[12:] - series_raw[:-12]

from statsmodels.tsa.stattools import kpss

def run_adf(series, label):
    r = adfuller(series[~np.isnan(series)], autolag='AIC')
    return {
        'Series':        label,
        'ADF Statistic': round(r[0], 4),
        'p-value':       round(r[1], 4),
        'Lags':          r[2],
        'Crit 5%':       round(r[4]['5%'], 4),
    }

def run_kpss(series, label):
    stat, p, nlags, cv = kpss(series[~np.isnan(series)], regression='c', nlags='auto')
    return {'Series': label, 'KPSS Statistic': round(stat,4),
            'p-value (≈)': round(p,4), 'Crit 5%': round(cv['5%'],4)}

adf_results = [
    run_adf(series_raw, 'Исходный ряд'),
    run_adf(series_d1,  'Первые разности'),
    run_adf(series_sd,  'Сезонные разности (лаг 12)'),
]
kpss_results = [
    run_kpss(series_raw, 'Исходный ряд'),
    run_kpss(series_d1,  'Первые разности'),
    run_kpss(series_sd,  'Сезонные разности (лаг 12)'),
]

df_adf = pd.DataFrame(adf_results)
df_adf.to_csv(f'{OUT_TABLES}/05_stationarity_adf_tests.csv', index=False)
print("ADF:")
display(df_adf)
print("\nKPSS:")
display(pd.DataFrame(kpss_results))


## 4. Автокорреляционные функции (ACF)

In [ ]:
K = 30
acf_raw   = acf_func(series_raw, nlags=K, fft=True)
acf_diff1 = acf_func(series_d1,  nlags=K, fft=True)
acf_sdiff = acf_func(series_sd,  nlags=K, fft=True)

print(f"✓ ACF рассчитаны для {K} лагов")
print(f"  Лаг-12 (исходный ряд): r = {acf_raw[12]:.4f}")
print(f"  Лаг-12 (перв. разности): r = {acf_diff1[12]:.4f}")


## 5. Модели тренда

In [ ]:
# Линейная модель на MA12 (2010–2019)
valid_pre = ~np.isnan(df['ma12']) & (df['year'] <= 2019)
t_pre = df.loc[valid_pre, 't'].values.astype(float)
y_pre = df.loc[valid_pre, 'ma12'].values
c_lin = np.polyfit(t_pre, y_pre, 1)
y_lin_pre = np.polyval(c_lin, t_pre)
r2_lin   = 1 - np.sum((y_pre - y_lin_pre)**2) / np.sum((y_pre - np.mean(y_pre))**2)
rmse_lin = np.sqrt(np.mean((y_pre - y_lin_pre)**2))
mape_lin = np.mean(np.abs((y_pre - y_lin_pre)/y_pre)) * 100

# Квадратичная модель на MA12 (весь период)
valid_all = ~np.isnan(df['ma12'])
t_all = df.loc[valid_all, 't'].values.astype(float)
y_all = df.loc[valid_all, 'ma12'].values
c_quad  = np.polyfit(t_all, y_all, 2)
y_quad  = np.polyval(c_quad, t_all)
r2_quad  = 1 - np.sum((y_all - y_quad)**2) / np.sum((y_all - np.mean(y_all))**2)
rmse_quad = np.sqrt(np.mean((y_all - y_quad)**2))
mape_quad = np.mean(np.abs((y_all - y_quad)/y_all)) * 100

df.loc[valid_all, 'trend_quad'] = y_quad
df.loc[valid_pre, 'trend_lin']  = y_lin_pre

# Расширение линейного тренда на весь период (для графика)
t_all_valid = df.loc[valid_all, 't'].values.astype(float)
y_lin_full  = np.polyval(c_lin, t_all_valid)

pd.DataFrame([
    {'Модель': 'Линейная (MA12, 2010–2019)', 'R²': round(r2_lin,4),
     'RMSE':   round(rmse_lin,4), 'MAPE, %': round(mape_lin,2)},
    {'Модель': 'Квадратичная (MA12, полный)', 'R²': round(r2_quad,4),
     'RMSE':   round(rmse_quad,4), 'MAPE, %': round(mape_quad,2)},
])


## 6. Аномалии и дополнительные тесты

In [ ]:
res_valid = df['residual'].dropna()
res_mean  = res_valid.mean()
res_std   = res_valid.std()
df['z_score']        = (df['residual'] - res_mean) / res_std
df['is_outlier_z']     = np.abs(df['z_score']) >= 2.0
df['is_outlier_covid'] = df['year'].isin([2020, 2021])
Q1  = df['arrivals_m'].quantile(0.25)
Q3  = df['arrivals_m'].quantile(0.75)
IQR = Q3 - Q1
df['is_outlier_iqr'] = (df['arrivals_m'] < Q1-1.5*IQR) | (df['arrivals_m'] > Q3+1.5*IQR)

# ── Тесты ─────────────────────────────────────────────────────────────────
resid_valid = df['residual'].dropna().values
sw_stat, sw_p = stats.shapiro(resid_valid)

diff_resid = np.diff(resid_valid)
dw_stat    = np.sum(diff_resid**2) / np.sum(resid_valid**2)

def turning_points(x):
    tp = 0
    for i in range(1, len(x)-1):
        if (x[i]>x[i-1] and x[i]>x[i+1]) or (x[i]<x[i-1] and x[i]<x[i+1]):
            tp += 1
    return tp

n_res  = len(resid_valid)
tp_obs = turning_points(resid_valid)
tp_exp = 2*(n_res-2)/3
tp_var = (16*n_res-29)/90
tp_z   = (tp_obs - tp_exp) / np.sqrt(tp_var)

pd.DataFrame([
    {'Критерий': 'Шапиро–Уилк',      'Статистика': f'W={sw_stat:.4f}', 'p-значение': f'{sw_p:.4f}',  'Вывод': 'нормальность отвергается'},
    {'Критерий': 'Дарбин–Уотсон',    'Статистика': f'd={dw_stat:.4f}', 'p-значение': '---',           'Вывод': 'положит. автокорреляция'},
    {'Критерий': 'Поворотные точки', 'Статистика': f'z={tp_z:.2f}',   'p-значение': '---',           'Вывод': 'случайность отвергается'},
])


## 7. Сохранение таблиц (CSV)

In [ ]:
df.to_csv(f'{OUT_TABLES}/01_monthly_arrivals_cleaned_derived.csv', index=False)

annual = df.groupby('year').agg(
    total_m=('arrivals_m','sum'), mean_m=('arrivals_m','mean'),
    max_m=('arrivals_m','max'),   min_m=('arrivals_m','min'),
    std_m=('arrivals_m','std')).reset_index()
annual.to_csv(f'{OUT_TABLES}/02_annual_summary.csv', index=False)

sp_profile = pd.DataFrame({'month': range(1,13), 'month_name': months_full_ru,
    'seasonal_coef': [seas_coefs[m] for m in range(1,13)]})
for y in [2010, 2019, 2022, 2023]:
    yr = df[df['year']==y].set_index('month')['arrivals_m']
    s  = yr.sum()
    sp_profile[f'pct_{y}'] = [yr.get(m, np.nan)/s*100 for m in range(1,13)]
sp_profile.to_csv(f'{OUT_TABLES}/03_seasonality_profile.csv', index=False)

df[['date','arrivals_m','ma12','seasonal','residual']].to_csv(
    f'{OUT_TABLES}/06_decomposition_additive.csv', index=False)

df[df['is_outlier_z']|df['is_outlier_covid']|df['is_outlier_iqr']][
    ['date','year','month','arrivals_m','z_score','is_outlier_z','is_outlier_covid','is_outlier_iqr']
].to_csv(f'{OUT_TABLES}/07_outliers.csv', index=False)

y_raw = df['arrivals_m'].values
def mae(y, yh):   return np.nanmean(np.abs(y - yh))
def rmse_f(y, yh): return np.sqrt(np.nanmean((y-yh)**2))
smooth_res = pd.DataFrame([
    {'Метод': 'MA(3)',         'MAE': mae(y_raw[2:], df['ma3'].values[2:]),      'RMSE': rmse_f(y_raw[2:], df['ma3'].values[2:])},
    {'Метод': 'MA(6)',         'MAE': mae(y_raw[5:], df['ma6'].values[5:]),      'RMSE': rmse_f(y_raw[5:], df['ma6'].values[5:])},
    {'Метод': 'MA(12) центр.', 'MAE': mae(y_raw[6:-6], df['ma12'].values[6:-6]), 'RMSE': rmse_f(y_raw[6:-6], df['ma12'].values[6:-6])},
    {'Метод': 'Exp(α=0.2)',    'MAE': mae(y_raw[1:], df['exp02'].values[1:]),    'RMSE': rmse_f(y_raw[1:], df['exp02'].values[1:])},
    {'Метод': 'Exp(α=0.6)',    'MAE': mae(y_raw[1:], df['exp06'].values[1:]),    'RMSE': rmse_f(y_raw[1:], df['exp06'].values[1:])},
])
smooth_res.to_csv(f'{OUT_TABLES}/08_model_summary.csv', index=False)

print("✓ Все CSV сохранены в", OUT_TABLES)
smooth_res


## 8. Графики
### Рис. 01 — Основной ряд

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
for ph, grp in df.groupby('phase'):
    ax.fill_between(grp['date'], 0, grp['arrivals_m'], alpha=0.15, color=phase_color[ph])
    ax.plot(grp['date'], grp['arrivals_m'], color=phase_color[ph], lw=1.5)

for y, m in [(2020,1), (2022,1)]:
    ax.axvline(pd.Timestamp(year=y,month=m,day=1), color='grey', ls='--', lw=0.9, alpha=0.65)

ax.annotate('Март 2020\nпандемия', xy=(pd.Timestamp('2020-03-01'), 0.263),
            xytext=(pd.Timestamp('2017-06-01'), 0.8),
            arrowprops=dict(arrowstyle='->', color='#D7191C'),
            fontsize=9, color='#D7191C')

legend_handles = [Patch(facecolor=phase_color[k], alpha=0.7, label=phase_label[k])
                  for k in ['growth','covid','recovery']]
ax.legend(handles=legend_handles, loc='upper left', framealpha=0.92)
ax.set_title('Ежемесячные международные туристские прибытия в Испанию, 2010–2023')
ax.set_ylabel('Прибытия, млн чел.')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}'))
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig01_monthly_series.pdf', bbox_inches='tight')
plt.show()


### Рис. 02 — Годовые итоги

In [ ]:
ann_tot = df.groupby('year')['arrivals_m'].sum()
ph_y    = {y: phase(y) for y in ann_tot.index}
col_bar = [phase_color[ph_y[y]] for y in ann_tot.index]

fig, ax = plt.subplots(figsize=(12,5))
bars = ax.bar(ann_tot.index, ann_tot.values, color=col_bar, edgecolor='white', lw=0.8, width=0.7)
for bar, val in zip(bars, ann_tot.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_title('Годовые итоги международных туристских прибытий в Испанию, 2010–2023')
ax.set_ylabel('Прибытия за год, млн чел.')
ax.set_xticks(ann_tot.index)
ax.set_xticklabels([str(y) for y in ann_tot.index], rotation=45)
legend_handles = [Patch(facecolor=phase_color[k], alpha=0.85, label=phase_label[k])
                  for k in ['growth','covid','recovery']]
ax.legend(handles=legend_handles, loc='upper left')
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig02_annual_totals.pdf', bbox_inches='tight')
plt.show()


### Рис. 03 — Сезонный профиль

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
sel_yrs  = [2010, 2019, 2022, 2023]
cols_s   = ['#A9C6DE', '#2C7BB6', '#73C374', '#1A9641']
lss_s    = ['--', '-', '--', '-']
mks_s    = ['o', 's', 'D', '^']
for y, col, ls, mk in zip(sel_yrs, cols_s, lss_s, mks_s):
    yr_dat = df[df['year']==y].set_index('month')['arrivals_m']
    vals   = [yr_dat.get(m, np.nan) for m in range(1,13)]
    ax.plot(range(1,13), vals, color=col, ls=ls, lw=2, marker=mk, ms=5, label=str(y))
ax.axvspan(6.5, 8.5, alpha=0.10, color='gold', label='Летний пик (июль–август)')
ax.set_xticks(range(1,13))
ax.set_xticklabels(months_ru)
ax.set_title('Сезонный профиль прибытий: 2010, 2019, 2022 и 2023')
ax.set_ylabel('Прибытия, млн чел.')
ax.legend(loc='upper left', framealpha=0.9)
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig03_seasonal_profile.pdf', bbox_inches='tight')
plt.show()


### Рис. 04 — Гистограмма

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
vals = df['arrivals_m'].values

ax = axes[0]
ax.hist(vals, bins=28, color='#2C7BB6', edgecolor='white', alpha=0.82)
ax.axvline(np.mean(vals),   color='red',    ls='--', lw=1.5, label=f'Среднее = {np.mean(vals):.2f}')
ax.axvline(np.median(vals), color='orange', ls='-.', lw=1.5, label=f'Медиана = {np.median(vals):.2f}')
ax.set_title('Гистограмма помесячных уровней ряда')
ax.set_xlabel('Прибытия, млн чел.'); ax.set_ylabel('Частота')
ax.legend()

ax2 = axes[1]
for ph, col in phase_color.items():
    vals_ph = df[df['phase']==ph]['arrivals_m'].values
    ax2.hist(vals_ph, bins=15, color=col, alpha=0.65, label=phase_label[ph])
ax2.set_title('Гистограмма по фазам')
ax2.set_xlabel('Прибытия, млн чел.'); ax2.set_ylabel('Частота')
ax2.legend(fontsize=8.5)
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig04_histogram.pdf', bbox_inches='tight')
plt.show()


### Рис. 05 — Коррелограммы (ACF)

In [ ]:
ci_raw  = 1.96 / np.sqrt(len(series_raw))
ci_d1   = 1.96 / np.sqrt(len(series_d1))
ci_sd   = 1.96 / np.sqrt(len(series_sd))

fig, axes = plt.subplots(3, 1, figsize=(14, 15))  # увеличен с (13,11) до (14,15)
configs = [
    (acf_raw,   ci_raw, '(а) Исходный ряд'),
    (acf_diff1, ci_d1,  '(б) Первые разности'),
    (acf_sdiff, ci_sd,  '(в) Сезонные разности (лаг 12)'),
]
for axt, (acf_v, ci, title) in zip(axes, configs):
    lags  = range(len(acf_v))
    col_b = ['#D7191C' if abs(v) > ci else '#2C7BB6' for v in acf_v]
    axt.bar(lags, acf_v, color=col_b, alpha=0.75, width=0.6)
    axt.axhline( ci, color='grey', ls='--', lw=1.2, alpha=0.8, label='95% CI')
    axt.axhline(-ci, color='grey', ls='--', lw=1.2, alpha=0.8)
    axt.axhline(0,   color='black', lw=0.8)
    for lag12 in [12, 24]:
        if lag12 < len(acf_v):
            axt.axvline(lag12, color='orange', ls=':', lw=1.3, alpha=0.75)
    axt.set_title(title, fontsize=13)
    axt.set_ylabel('r(k)', fontsize=11)
    axt.set_xlabel('Лаг k', fontsize=11)
    axt.set_xlim(-0.5, len(acf_v)-0.5)
    axt.tick_params(labelsize=10)
plt.suptitle('Автокорреляционные функции: исходный ряд, первые разности, сезонные разности',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout(h_pad=3.5)
fig.savefig(f'{OUT_ASSETS}/fig05_correlogram.pdf', bbox_inches='tight')
plt.show()


### Рис. 13 — ACF + PACF первых разностей

In [ ]:
from statsmodels.tsa.stattools import pacf as pacf_func

K = 30
pacf_d1 = pacf_func(series_d1, nlags=min(K, len(series_d1)//2 - 1), method='ywm')
ci_d1_pacf = 1.96 / np.sqrt(len(series_d1))

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

ax_acf = axes[0]
lags_acf = range(len(acf_diff1))
col_acf = ['#D7191C' if abs(v) > ci_d1 else '#2C7BB6' for v in acf_diff1]
ax_acf.bar(lags_acf, acf_diff1, color=col_acf, alpha=0.75, width=0.6)
ax_acf.axhline( ci_d1, color='grey', ls='--', lw=1.2, alpha=0.8, label='95% CI')
ax_acf.axhline(-ci_d1, color='grey', ls='--', lw=1.2, alpha=0.8)
ax_acf.axhline(0, color='black', lw=0.8)
for lag12 in [12, 24]:
    ax_acf.axvline(lag12, color='orange', ls=':', lw=1.3, alpha=0.75)
ax_acf.set_title('(а) ACF первых разностей', fontsize=13)
ax_acf.set_ylabel('r(k)', fontsize=11)
ax_acf.set_xlabel('Лаг k', fontsize=11)
ax_acf.set_xlim(-0.5, K + 0.5)
ax_acf.legend()

ax_pacf = axes[1]
lags_pacf = range(len(pacf_d1))
col_pacf = ['#D7191C' if abs(v) > ci_d1_pacf else '#2C7BB6' for v in pacf_d1]
ax_pacf.bar(lags_pacf, pacf_d1, color=col_pacf, alpha=0.75, width=0.6)
ax_pacf.axhline( ci_d1_pacf, color='grey', ls='--', lw=1.2, alpha=0.8, label='95% CI')
ax_pacf.axhline(-ci_d1_pacf, color='grey', ls='--', lw=1.2, alpha=0.8)
ax_pacf.axhline(0, color='black', lw=0.8)
for lag12 in [12, 24]:
    if lag12 < len(pacf_d1):
        ax_pacf.axvline(lag12, color='orange', ls=':', lw=1.3, alpha=0.75)
ax_pacf.set_title('(б) PACF первых разностей', fontsize=13)
ax_pacf.set_ylabel('φ(k)', fontsize=11)
ax_pacf.set_xlabel('Лаг k', fontsize=11)
ax_pacf.set_xlim(-0.5, len(pacf_d1) - 0.5)
ax_pacf.legend()

plt.suptitle('Идентификация зависимости: ACF и PACF первых разностей ряда',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout(h_pad=3.5)
fig.savefig(f'{OUT_ASSETS}/fig13_pacf.pdf', bbox_inches='tight')
plt.show()


### Рис. 06 — Декомпозиция

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14,12), sharex=True)
comps = [
    (df['arrivals_m'], 'Исходный ряд',         C_RAW,   False),
    (df['ma12'],       'Тренд-цикл (MA12)',     C_TREND, False),
    (df['seasonal'],   'Сезонная компонента',   C_SEAS,  True),
    (df['residual'],   'Остаточная компонента', C_RESID, True),
]
for ax_j, (series, title, color, bar) in zip(axes, comps):
    if bar:
        ax_j.bar(df['date'], series, color=color, alpha=0.7, width=20)
        ax_j.axhline(0, color='black', lw=0.7)
    else:
        ax_j.plot(df['date'], series, color=color, lw=1.5)
    ax_j.set_ylabel('млн чел.', fontsize=9)
    ax_j.set_title(title, fontsize=11)
plt.suptitle('Аддитивная декомпозиция временного ряда: Испания, 2010–2023',
             fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig06_decomposition.pdf', bbox_inches='tight')
plt.show()


### Рис. 07 — Скользящие средние

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(df['date'], df['arrivals_m'], color='#C0C7D4', lw=1.2, alpha=0.8, label='Исходный ряд')
ax.plot(df['date'], df['ma3'],  color=C_MA3,  lw=1.8, label='MA(3)')
ax.plot(df['date'], df['ma6'],  color=C_MA6,  lw=1.8, label='MA(6)')
ax.plot(df['date'], df['ma12'], color=C_MA12, lw=2.2, label='MA(12) центр.')
ax.set_title('Сглаживание ряда скользящими средними: MA(3), MA(6), MA(12)')
ax.set_ylabel('Прибытия, млн чел.')
ax.legend(loc='upper left', framealpha=0.9)
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig07_moving_averages.pdf', bbox_inches='tight')
plt.show()


### Рис. 08 — Аномалии

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14,9), sharex=True)

ax0 = axes[0]
ax0.plot(df['date'], df['arrivals_m'], color='#95A5A6', lw=1.3, alpha=0.85, label='Ряд')
covid_mask = df['is_outlier_covid']
ax0.fill_between(df['date'], 0, df['arrivals_m'].max()*1.08,
                 where=covid_mask, alpha=0.12, color='red', label='COVID-период (2020–2021)')
z_mask = df['is_outlier_z']
ax0.scatter(df.loc[z_mask,'date'], df.loc[z_mask,'arrivals_m'],
            color='#C0392B', zorder=5, s=65, marker='v', label='|z| ≥ 2 (остатки)')
ax0.set_ylabel('Прибытия, млн чел.')
ax0.set_title('Аномальные уровни ряда: COVID-период и z-критерий по остаткам')
ax0.legend()

ax1 = axes[1]
vz = df[~df['z_score'].isna()]
bar_col = ['#C0392B' if abs(z) >= 2 else '#2C7BB6' for z in vz['z_score']]
ax1.bar(vz['date'], vz['z_score'], color=bar_col, alpha=0.75, width=20)
ax1.axhline( 2, color='#E74C3C', ls='--', lw=1.2, label='±2σ (порог)')
ax1.axhline(-2, color='#E74C3C', ls='--', lw=1.2)
ax1.axhline(0,  color='black',   lw=0.8)
ax1.set_ylabel('z-оценка остатка')
ax1.set_title('Стандартизованная остаточная компонента')
ax1.legend()
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig08_outliers.pdf', bbox_inches='tight')
plt.show()


### Рис. 09 — Модели тренда

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5.5))

ax0 = axes[0]
ax0.plot(df['date'], df['arrivals_m'], color='#C0C7D4', lw=1.0, alpha=0.7, label='Исходный ряд')
ax0.plot(df.loc[valid_all,'date'], df.loc[valid_all,'ma12'],
         color='#2C3E50', lw=1.5, alpha=0.6, label='MA12 (тренд)')
ax0.plot(df.loc[valid_pre,'date'], y_lin_pre, color=C_LIN, lw=2.5,
         label=f'Линейный тренд (2010–2019)\nR²={r2_lin:.3f}, MAPE={mape_lin:.1f}%')
ax0.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2023-12-31'),
            alpha=0.07, color='red', label='Вне области применения')
ax0.set_title('Линейная модель (область 2010–2019)')
ax0.set_ylabel('Прибытия, млн чел.')
ax0.legend(fontsize=8.5, loc='upper left')

ax1 = axes[1]
ax1.plot(df['date'], df['arrivals_m'], color='#C0C7D4', lw=1.0, alpha=0.7, label='Исходный ряд')
ax1.plot(df.loc[valid_all,'date'], df.loc[valid_all,'ma12'],
         color='#2C3E50', lw=1.5, alpha=0.6, label='MA12 (тренд)')
ax1.plot(df.loc[valid_all,'date'], y_quad, color=C_QUAD, lw=2.5,
         label=f'Квадратичный тренд (полный)\nR²={r2_quad:.3f}, MAPE={mape_quad:.1f}%')
ax1.set_title('Квадратичная модель (весь период)')
ax1.set_ylabel('Прибытия, млн чел.')
ax1.legend(fontsize=8.5, loc='upper left')

plt.suptitle('Аппроксимация тренда: линейная vs. квадратичная модели', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig09_trend_models.pdf', bbox_inches='tight')
plt.show()


### Рис. 10 — Диагностика остатков

In [ ]:
fig = plt.figure(figsize=(14,6))
gs  = GridSpec(1, 2, figure=fig, width_ratios=[2,1])

ax_l = fig.add_subplot(gs[0])
vres = df[~df['residual'].isna()]
bar_col2 = ['#C0392B' if abs(z) >= 2 else '#2C7BB6'
            for z in vres['z_score'].fillna(0)]
ax_l.bar(vres['date'], vres['residual'], color=bar_col2, alpha=0.75, width=20)
ax_l.axhline( 2*res_std, color='grey', ls='--', lw=1.0, alpha=0.7, label='±2σ')
ax_l.axhline(-2*res_std, color='grey', ls='--', lw=1.0, alpha=0.7)
ax_l.axhline(0, color='black', lw=0.8)
ax_l.set_title('Остаточная компонента декомпозиции')
ax_l.set_ylabel('Остаток, млн чел.')
ax_l.legend()

ax_r = fig.add_subplot(gs[1])
(osm, osr), (slope, intercept, _r) = stats.probplot(resid_valid, dist='norm')
ax_r.scatter(osm, osr, color='#2C7BB6', alpha=0.65, s=20)
ax_r.plot([osm.min(), osm.max()],
          [slope*osm.min()+intercept, slope*osm.max()+intercept],
          color='#E74C3C', lw=1.5)
ax_r.set_title('Q–Q нормальности остатков')
ax_r.set_xlabel('Теоретические квантили'); ax_r.set_ylabel('Эмпирические квантили')

plt.suptitle(f'Диагностика остатков | Дарбин–Уотсон d={dw_stat:.3f} | '
             f'Шапиро–Уилк W={sw_stat:.4f}, p={sw_p:.4f}',
             fontsize=10, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig10_residuals.pdf', bbox_inches='tight')
plt.show()


### Рис. 11 — Экспоненциальное сглаживание

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(df['date'], df['arrivals_m'], color='#C0C7D4', lw=1.2, alpha=0.8, label='Исходный ряд')
ax.plot(df['date'], df['exp02'], color=C_EXP_LOW,  lw=2.0, label='Браун α=0.2 (сглаженный)')
ax.plot(df['date'], df['exp06'], color=C_EXP_HIGH, lw=2.0, ls='--', label='Браун α=0.6 (адаптивный)')
ax.set_title('Экспоненциальное сглаживание Брауна: α=0.2 и α=0.6')
ax.set_ylabel('Прибытия, млн чел.')
ax.legend(loc='upper left', framealpha=0.9)
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig11_exp_smoothing.pdf', bbox_inches='tight')
plt.show()


### Рис. 12 — Тепловая карта месяц × год

In [ ]:
from matplotlib.patches import Rectangle

pivot = df.pivot(index='month', columns='year', values='arrivals_m')

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', origin='upper')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(y) for y in pivot.columns], rotation=45, ha='right', fontsize=9.5)
ax.set_yticks(range(12))
ax.set_yticklabels(months_full_ru, fontsize=9.5)

for yi in range(12):
    for xi in range(len(pivot.columns)):
        val = pivot.values[yi, xi]
        if not np.isnan(val):
            text_color = 'white' if val > 7.0 else 'black'
            ax.text(xi, yi, f'{val:.1f}', ha='center', va='center',
                    fontsize=7.5, color=text_color, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.01)
cbar.set_label('Прибытия, млн чел.', fontsize=10)

rect = Rectangle((-0.5 + list(pivot.columns).index(2020), -0.5), 2, 12,
                 linewidth=2.5, edgecolor='#D7191C', facecolor='none',
                 linestyle='--', zorder=5)
ax.add_patch(rect)

ax.set_title('Тепловая карта: международные туристские прибытия в Испанию (млн чел., месяц × год)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig12_heatmap.pdf', bbox_inches='tight')
plt.show()


### Рис. 14 — Прогноз на 2024 год (Хольт–Уинтерс)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing as ETS

ts_hw = pd.Series(df['arrivals_m'].values,
                  index=pd.date_range('2010-01', periods=168, freq='MS'))
model_hw = ETS(ts_hw, trend='add', seasonal='add', seasonal_periods=12, damped_trend=False)
fit_hw   = model_hw.fit(optimized=True)

h = 12
fc_hw  = fit_hw.forecast(h)
sim    = fit_hw.simulate(h, repetitions=500, error='add', random_errors='bootstrap')
fc_lo  = np.percentile(sim, 2.5, axis=1)
fc_hi  = np.percentile(sim, 97.5, axis=1)
future_dates = pd.date_range('2024-01', periods=h, freq='MS')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ts_hw.index, ts_hw.values, color='#2C3E50', lw=1.3, alpha=0.7, label='Исходный ряд (2010–2023)')
ax.plot(fit_hw.fittedvalues.index, fit_hw.fittedvalues.values,
        color='#2C7BB6', lw=1.5, ls='--', alpha=0.6, label='Подгонка модели HW')
ax.plot(future_dates, fc_hw.values, color='#E74C3C', lw=2.5, marker='o', ms=5,
        label=f'Прогноз HW на 2024 (Σ={fc_hw.values.sum():.1f} млн)')
ax.fill_between(future_dates, fc_lo, fc_hi,
                color='#E74C3C', alpha=0.18, label='95% ДИ (bootstrap)')
ax.axvline(pd.Timestamp('2024-01-01'), color='grey', ls=':', lw=1.2)
ax.set_title('Прогноз международных туристских прибытий в Испанию на 2024 год\n'
             '(модель Хольта–Уинтерса, аддитивная сезонность)')
ax.set_ylabel('Прибытия, млн чел.')
ax.legend(loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}'))
fig.tight_layout()
fig.savefig(f'{OUT_ASSETS}/fig14_forecast.pdf', bbox_inches='tight')
plt.show()

print(f"Прогнозный годовой итог 2024: {fc_hw.values.sum()*1000:.0f} тыс. прибытий")


## 9. Генерация LaTeX-таблиц

In [ ]:
n_obs    = len(df)
arr_mean = df['arrivals_m'].mean()
arr_med  = df['arrivals_m'].median()
arr_std  = df['arrivals_m'].std()
arr_min  = df['arrivals_m'].min()
arr_max  = df['arrivals_m'].max()
arr_cv   = arr_std / arr_mean * 100

# tab_data_desc
with open(f'{OUT_GEN}/tab_data_desc.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Основные характеристики временного ряда}
\label{tab:data_desc}
\begin{tabular}{lc}
\toprule
Характеристика & Значение \\
\midrule
Период & январь 2010 --- декабрь 2023 \\
Число наблюдений $n$ & """ + str(n_obs) + r""" \\
Единица измерения & тыс. прибытий / месяц \\
Среднее $\bar{x}$ & """ + f"{arr_mean*1000:.0f}" + r"""\,тыс. \\
Медиана & """ + f"{arr_med*1000:.0f}" + r"""\,тыс. \\
Стандартное отклонение $s$ & """ + f"{arr_std*1000:.0f}" + r"""\,тыс. \\
Коэффициент вариации & """ + f"{arr_cv:.1f}" + r"""\,\% \\
Минимум & """ + f"{arr_min*1000:.0f}" + r"""\,тыс. (апрель 2020) \\
Максимум & """ + f"{arr_max*1000:.0f}" + r"""\,тыс. (август 2023) \\
Источник & INE FRONTUR \\
\bottomrule
\end{tabular}
\end{table}
""")

# tab_phases
with open(f'{OUT_GEN}/tab_phases.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Фазы анализируемого периода}
\label{tab:phases}
\begin{tabular}{llcc}
\toprule
Фаза & Период & Годовой итог, млн & Динамика \\
\midrule
Устойчивый рост & 2010--2019 & 52,7 --- 83,7 & +59\,\% за десятилетие \\
COVID-шок & 2020--2021 & 19,0 / 31,2 & $-77\,\%$ / $-63\,\%$ к 2019 \\
Восстановление & 2022--2023 & 71,7 / 85,1 & $-14\,\%$ / $+1{,}7\,\%$ к 2019 \\
\bottomrule
\end{tabular}
\end{table}
""")

# tab_adf (ADF + KPSS combined)
with open(f'{OUT_GEN}/tab_adf.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Результаты тестов ADF и KPSS на стационарность}
\label{tab:adf}
{\small
\setlength{\tabcolsep}{3.5pt}
\begin{tabular}{lcccccc}
\toprule
\multirow{2}{*}{Ряд} & \multicolumn{2}{c}{ADF} & & \multicolumn{2}{c}{KPSS} & \multirow{2}{*}{Вывод} \\
\cmidrule{2-3}\cmidrule{5-6}
 & Стат. & $p$ & & Стат. & $p$ & \\
\midrule
""")
    for adf_r, kpss_r in zip(adf_results, kpss_results):
        adf_nonstat  = adf_r['p-value'] > 0.05
        kpss_nonstat = kpss_r['p-value (≈)'] < 0.05
        if adf_nonstat and not kpss_nonstat:
            verdict = r'нестационарен$^*$'
        elif not adf_nonstat and not kpss_nonstat:
            verdict = 'стационарен'
        else:
            verdict = 'нестационарен'
        f.write(f"{adf_r['Series']} & {adf_r['ADF Statistic']} & {adf_r['p-value']} "
                f"& & {kpss_r['KPSS Statistic']} & {kpss_r['p-value (≈)']} & {verdict} \\\n")
    f.write(r"""\bottomrule
\end{tabular}}
\medskip\par
{\footnotesize
ADF: $H_0$ --- единичный корень; $p>0{,}05$ $\Rightarrow$ нестационарен.
KPSS: $H_0$ --- стационарность; $p<0{,}05$ $\Rightarrow$ нестационарен.
$^*$Оба теста согласованы: ADF не отвергает нестационарность,
KPSS не отвергает стационарность --- пограничный случай (COVID-шок).}
\end{table}
""")

# tab_trend
with open(f'{OUT_GEN}/tab_trend.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Параметры и качество моделей тренда}
\label{tab:trend}
\begin{tabular}{llccc}
\toprule
Модель & Область & $R^2$ & RMSE, млн & MAPE, \% \\
\midrule
Линейная & MA12, 2010--2019 & """ + f"{r2_lin:.3f}" + r""" & """ + f"{rmse_lin:.3f}" + r""" & """ + f"{mape_lin:.1f}" + r""" \\
Квадратичная & MA12, 2010--2023 & """ + f"{r2_quad:.3f}" + r""" & """ + f"{rmse_quad:.3f}" + r""" & """ + f"{mape_quad:.1f}" + r""" \\
\bottomrule
\end{tabular}
\end{table}
""")

# tab_smooth
with open(f'{OUT_GEN}/tab_smooth.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Сравнение методов сглаживания}
\label{tab:smooth}
\begin{tabular}{lcc}
\toprule
Метод & MAE, млн & RMSE, млн \\
\midrule
""")
    for _, row in smooth_res.iterrows():
        f.write(f"{row['Метод']} & {row['MAE']:.3f} & {row['RMSE']:.3f} \\\n")
    f.write(r"""\bottomrule
\end{tabular}
\end{table}
""")

# tab_extra_tests
with open(f'{OUT_GEN}/tab_extra_tests.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Дополнительные критерии оценки временного ряда}
\label{tab:extra_tests}
\begin{tabular}{llll}
\toprule
Критерий & Статистика & $p$-значение & Вывод \\
\midrule
Шапиро--Уилк & $W=""" + f"{sw_stat:.4f}" + r"""$ & """ + f"{sw_p:.4f}" + r""" & нормальность отвергается \\
Дарбин--Уотсон & $d=""" + f"{dw_stat:.4f}" + r"""$ & --- & положит. автокорреляция \\
Поворотные точки & $z=""" + f"{tp_z:.2f}" + r"""$ & --- & случайность отвергается \\
\bottomrule
\end{tabular}
\end{table}
""")

# tab_seasonal (с 95% доверительными интервалами)
with open(f'{OUT_GEN}/tab_seasonal.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Сезонные коэффициенты аддитивной декомпозиции с 95\,\% ДИ (тыс.~чел.)}
\label{tab:seasonal}
{\small
\begin{tabular}{lrrr}
\toprule
Месяц & Коэффициент & НГ 95\,\%~ДИ & ВГ 95\,\%~ДИ \\
\midrule
""")
    for m in range(1,13):
        c  = seas_coefs[m] * 1000
        hw = seas_ci[m]    * 1000
        lo = c - hw
        hi = c + hw
        f.write(f"{months_full_ru[m-1]} & {c:+.0f} & {lo:+.0f} & {hi:+.0f} \\\n")
    f.write(r"""\bottomrule
\end{tabular}}
\end{table}
""")

print("✓ Все LaTeX-таблицы сохранены в", OUT_GEN)


# tab_forecast
from statsmodels.tsa.holtwinters import ExponentialSmoothing as ETS_gen
ts_gen = pd.Series(df['arrivals_m'].values,
                   index=pd.date_range('2010-01', periods=168, freq='MS'))
fit_gen = ETS_gen(ts_gen, trend='add', seasonal='add', seasonal_periods=12).fit(optimized=True)
fc_gen  = fit_gen.forecast(12)
sim_gen = fit_gen.simulate(12, repetitions=500, error='add', random_errors='bootstrap')
fc_lo_g = np.percentile(sim_gen, 2.5, axis=1)
fc_hi_g = np.percentile(sim_gen, 97.5, axis=1)
months_full_ru_g = ['Январь','Февраль','Март','Апрель','Май','Июнь',
                    'Июль','Август','Сентябрь','Октябрь','Ноябрь','Декабрь']
with open(f'{OUT_GEN}/tab_forecast.tex','w') as f:
    f.write(r"""\begin{table}[h!]
\centering
\caption{Прогноз международных туристских прибытий в Испанию на 2024 год (Хольт--Уинтерс)}
\label{tab:forecast}
{\small
\begin{tabular}{lrrr}
\toprule
Месяц & Прогноз, тыс. & НГ 95\,\%~ДИ, тыс. & ВГ 95\,\%~ДИ, тыс. \\
\midrule
""")
    for i in range(12):
        f.write(f"{months_full_ru_g[i]} & {fc_gen.values[i]*1000:.0f} & {fc_lo_g[i]*1000:.0f} & {fc_hi_g[i]*1000:.0f} \\\n")
    f.write(f"\\midrule\nГодовой итог & {fc_gen.values.sum()*1000:.0f} & --- & --- \\\n")
    f.write(r"""\bottomrule
\end{tabular}}
\end{table}
""")
print(f"✓ tab_forecast: прогноз 2024 = {fc_gen.values.sum()*1000:.0f} тыс.")